# Baseline: ConvNeXt JEPA for Active Matter

Standalone reimplementation of the baseline from [arXiv:2603.13227](https://arxiv.org/abs/2603.13227)
([GitHub](https://github.com/helenqu/physical-representation-learning)).

## Method
- **Encoder**: ConvNeXt-style 3D CNN (dims [16,32,64,128,128]) that maps `(B, 11, 16, 224, 224)` → `(B, 128, 14, 14)`
- **Predictor**: Small 2D CNN that maps context embedding → predicted target embedding
- **Loss**: VICReg (invariance + variance + covariance)
- **Task**: Given 16 context frames, predict the representation of the *next* 16 frames

## Evaluation
- Linear probe (Ridge regression) on frozen features → MSE for z-scored α, ζ
- kNN regression on frozen features → MSE for z-scored α, ζ


In [1]:
# !pip install torch torchvision h5py scikit-learn wandb matplotlib timm


In [2]:
import os, math, random, re, copy, gc
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict
from collections import OrderedDict, defaultdict
import weakref

import numpy as np
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

import matplotlib.pyplot as plt

try:
    from timm.models.layers import DropPath
except ImportError:
    class DropPath(nn.Module):
        def __init__(self, drop_prob=0.):
            super().__init__()
            self.drop_prob = drop_prob
        def forward(self, x):
            if not self.training or self.drop_prob == 0.:
                return x
            keep = torch.rand(x.shape[0], 1, *([1]*(x.ndim-2)), device=x.device) > self.drop_prob
            return x * keep / (1 - self.drop_prob)

try:
    import wandb; HAS_WANDB = True
except ImportError:
    HAS_WANDB = False

print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


PyTorch 2.8.0+cu128, CUDA: True
GPU: NVIDIA A100-SXM4-40GB


## Configuration


In [3]:
NETID = os.environ.get('NETID', 'kk5047')

@dataclass
class Config:
    # ── Data ─────────────────────────────────────────
    data_dir: str            = f'/scratch/{NETID}/dl-proj/data/active_matter/data'
    num_frames: int          = 16
    num_channels: int        = 11
    resolution: int          = 224
    temporal_stride: int     = 1     # offset=1 as in baseline config
    noise_std: float         = 1.0   # Gaussian noise during training

    # ── Model (conv_small from baseline) ─────────────
    dims: Tuple[int, ...]    = (16, 32, 64, 128, 128)
    num_res_blocks: Tuple[int, ...] = (3, 3, 3, 9, 3)

    # ── VICReg loss coefficients ─────────────────────
    sim_coeff: float         = 2.0
    std_coeff: float         = 40.0
    cov_coeff: float         = 2.0

    # ── Training ─────────────────────────────────────
    batch_size: int          = 8
    lr: float                = 1e-3
    min_lr: float            = 1e-6
    weight_decay: float      = 0.05
    epochs: int              = 30
    warmup_epochs: int       = 2
    clip_grad: float         = 1.0
    target_global_batch_size: int = 256

    # ── Eval ─────────────────────────────────────────
    eval_interval: int       = 5
    knn_k: int               = 20

    # ── Checkpoint ───────────────────────────────────
    checkpoint_dir: str      = f'/scratch/{NETID}/dl-proj/checkpoints/baseline_jepa'
    save_interval: int       = 2

    # ── Misc ─────────────────────────────────────────
    seed: int                = 42
    device: str              = 'cuda' if torch.cuda.is_available() else 'cpu'
    num_workers: int         = 4
    use_wandb: bool          = True
    wandb_project: str       = 'dl-research-project'

cfg = Config()

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
print(f'Config ready. Device: {cfg.device}')


Config ready. Device: cuda


## Dataset

Each sample is a `(context, target)` pair of 16 consecutive frames each.  
Index = `(file, trajectory, t0)` where context = `[t0, t0+16)`, target = `[t0+16, t0+32)`.  
175 trajectories × 50 valid starts (stride=1) = 8,750 training samples.


In [4]:
FIELD_PATHS = [
    ('t0_fields/concentration', ()),      # scalar  -> 1 ch
    ('t1_fields/velocity',      (2,)),    # vector  -> 2 ch
    ('t2_fields/D',             (2, 2)),  # tensor  -> 4 ch
    ('t2_fields/E',             (2, 2)),  # tensor  -> 4 ch
]                                          # Total: 11 ch


class ActiveMatterJEPA(Dataset):
    """
    Yields (context, target) pairs for the temporal-prediction JEPA task.
    context = frames [t0, t0+F), target = frames [t0+F, t0+2F).
    """

    def __init__(
        self,
        split: str,
        data_dir: str,
        num_frames: int      = 16,
        resolution: int      = 224,
        stride: int          = 1,
        noise_std: float     = 0.0,
        max_open_files: int  = 6,
    ):
        if split == 'val': split = 'valid'
        if split == 'validation': split = 'valid'
        self.data_dir    = Path(data_dir) / split
        self.F           = num_frames
        self.resolution  = (resolution, resolution)
        self.noise_std   = noise_std
        self._max_open   = max_open_files
        self._open       = None                      # per-worker LRU cache

        # Discover field schema from first file
        self.files = sorted(self.data_dir.glob('*.hdf5'))
        assert self.files, f'No .hdf5 files in {self.data_dir}'
        self._build_field_schema(self.files[0])

        # Build flat index: (filename, trajectory_id, t0)
        self.index = []
        self.file_params = {}   # filename -> [alpha, zeta]
        F = self.F
        for fpath in self.files:
            with h5py.File(fpath, 'r') as f:
                n_traj = f[self._field_paths[0]].shape[0]
                T      = f[self._field_paths[0]].shape[1]
                max_t0 = T - 2 * F
                if max_t0 < 0:
                    continue
                for ti in range(n_traj):
                    for t0 in range(0, max_t0 + 1, stride):
                        self.index.append((fpath.name, ti, t0))
                # Physical params (exclude L)
                self.file_params[fpath.name] = [
                    f['scalars'][k][()].item()
                    for k in sorted(f['scalars'].keys()) if k != 'L'
                ]

        print(f'{split}: {len(self.index):,} samples from {len(self.files)} files')

    def _build_field_schema(self, sample_path):
        self._field_paths = []
        self._comp_shapes = []
        self._d_sizes     = []
        with h5py.File(sample_path, 'r') as f:
            for fpath, comp in FIELD_PATHS:
                if fpath in f:
                    self._field_paths.append(fpath)
                    self._comp_shapes.append(comp)
                    self._d_sizes.append(int(np.prod(comp)) if comp else 1)
            _, _, H, W = f[self._field_paths[0]].shape[:4]
        self._C_total = sum(self._d_sizes)
        self._spatial = (H, W)

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        fname, ti, t0 = self.index[i]
        F = self.F
        H, W = self._spatial
        C = self._C_total

        fh, _ = self._open_file(fname)

        ctx = np.empty((F, H, W, C), dtype=np.float32)
        tgt = np.empty((F, H, W, C), dtype=np.float32)

        c0 = 0
        for fpath, comp, dsize in zip(self._field_paths, self._comp_shapes, self._d_sizes):
            ds = fh[fpath]
            sel = (ti, slice(t0, t0 + 2*F), slice(None), slice(None))
            sel += tuple(slice(None) for _ in comp)
            buf = ds[sel].astype(np.float32)                   # (2F, H, W, *comp)
            view = buf.reshape(2*F, H, W, dsize)
            c1 = c0 + dsize
            ctx[..., c0:c1] = view[:F]
            tgt[..., c0:c1] = view[F:]
            c0 = c1

        # (F, H, W, C) -> (C, F, H, W)
        ctx_t = torch.from_numpy(ctx).permute(3, 0, 1, 2).contiguous()
        tgt_t = torch.from_numpy(tgt).permute(3, 0, 1, 2).contiguous()

        # Resize 256 -> 224 (treat T as batch dim for 2D interpolation)
        if (H, W) != self.resolution:
            ctx_t = ctx_t.permute(1, 0, 2, 3)  # (T, C, H, W)
            ctx_t = F_func.interpolate(ctx_t, size=self.resolution,
                                        mode='bilinear', align_corners=False)
            ctx_t = ctx_t.permute(1, 0, 2, 3)  # (C, T, H, W)

            tgt_t = tgt_t.permute(1, 0, 2, 3)
            tgt_t = F_func.interpolate(tgt_t, size=self.resolution,
                                        mode='bilinear', align_corners=False)
            tgt_t = tgt_t.permute(1, 0, 2, 3)


        # Gaussian noise
        if self.noise_std > 0 and self.training_mode:
            ctx_t = ctx_t + torch.randn_like(ctx_t) * self.noise_std
            tgt_t = tgt_t + torch.randn_like(tgt_t) * self.noise_std

        params = torch.tensor(self.file_params[fname], dtype=torch.float32)

        return {
            'context':         ctx_t,    # (C, F, H, W) = (11, 16, 224, 224)
            'target':          tgt_t,    # (C, F, H, W)
            'physical_params': params,   # [alpha, zeta]
        }

    # ── Per-worker LRU file cache ────────────────────
    @property
    def training_mode(self):
        return self.noise_std > 0

    def _ensure_state(self):
        if self._open is None:
            self._open = OrderedDict()
            weakref.finalize(self, self._close_all)

    def _close_all(self):
        if self._open:
            for fh, _ in self._open.values():
                try: fh.close()
                except: pass
            self._open.clear()

    def _open_file(self, fname):
        self._ensure_state()
        if fname in self._open:
            val = self._open.pop(fname)
            self._open[fname] = val
            return val
        while len(self._open) >= self._max_open:
            _, (old, _) = self._open.popitem(last=False)
            try: old.close()
            except: pass
        fh = h5py.File(self.data_dir / fname, 'r', swmr=True)
        st = {}
        self._open[fname] = (fh, st)
        return fh, st

    def __getstate__(self):
        st = self.__dict__.copy()
        st['_open'] = None
        return st


# Alias F to avoid conflict with torch.nn.functional
F_func = F


def build_dataloaders(cfg):
    train_ds = ActiveMatterJEPA(
        'train', cfg.data_dir, cfg.num_frames, cfg.resolution,
        stride=cfg.temporal_stride, noise_std=cfg.noise_std,
    )
    val_ds = ActiveMatterJEPA(
        'valid', cfg.data_dir, cfg.num_frames, cfg.resolution,
        stride=cfg.temporal_stride, noise_std=0.0,
    )
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, pin_memory=True, drop_last=True,
        persistent_workers=(cfg.num_workers > 0),
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, pin_memory=True,
    )
    return train_loader, val_loader

print('Dataset defined.')


Dataset defined.


## Model: ConvNeXt Encoder + Predictor


In [5]:
class LayerNorm(nn.Module):
    """Channel-first or channel-last LayerNorm."""
    def __init__(self, dim, eps=1e-6, data_format='channels_last'):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias   = nn.Parameter(torch.zeros(dim))
        self.eps    = eps
        self.data_format = data_format

    def forward(self, x):
        if self.data_format == 'channels_last':
            return F.layer_norm(x, (self.weight.shape[0],), self.weight, self.bias, self.eps)
        # channels_first
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        shape = [1] * x.ndim
        shape[1] = -1
        return self.weight.view(shape) * x + self.bias.view(shape)


class ResidualBlock(nn.Module):
    """ConvNeXt-style residual block (depthwise conv + MLP)."""
    def __init__(self, dim, num_spatial_dims=3, layer_scale_init=1e-6, drop_path=0.):
        super().__init__()
        Conv = nn.Conv3d if num_spatial_dims == 3 else nn.Conv2d
        self.ndim = num_spatial_dims
        self.conv   = Conv(dim, dim, kernel_size=7, padding=3, groups=dim)
        self.norm   = LayerNorm(dim, data_format='channels_last')
        self.pwconv1 = nn.Linear(dim, 4 * dim)
        self.act     = nn.GELU()
        self.pwconv2 = nn.Linear(4 * dim, dim)
        self.gamma   = nn.Parameter(layer_scale_init * torch.ones(dim)) if layer_scale_init > 0 else None
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        shortcut = x
        x = self.conv(x)
        # channels first -> channels last
        if self.ndim == 3:
            x = x.permute(0, 2, 3, 4, 1)
        else:
            x = x.permute(0, 2, 3, 1)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.pwconv2(x)
        if self.gamma is not None:
            x = self.gamma * x
        # channels last -> channels first
        if self.ndim == 3:
            x = x.permute(0, 4, 1, 2, 3)
        else:
            x = x.permute(0, 3, 1, 2)
        return shortcut + self.drop_path(x)


class ConvEncoder(nn.Module):
    """
    ConvNeXt-style 3D encoder.
    Input:  (B, C=11, T=16, H=224, W=224)
    Output: (B, dims[-1], H/16, W/16) = (B, 128, 14, 14)

    4 downsampling stages, each halving all dims.
    After stage 3, temporal dim collapses to 1 -> squeeze to 2D.
    Last stage uses 2D convs/blocks.
    """
    def __init__(self, in_chans=11, dims=(16,32,64,128,128), num_res_blocks=(3,3,3,9,3), num_frames=16):
        super().__init__()
        assert num_frames == 16, 'This encoder assumes num_frames=16'

        # Stem: spatial-only conv, preserves all dims
        stem = nn.Sequential(
            nn.Conv3d(in_chans, dims[0], kernel_size=(1, 4, 4), padding='same'),
            LayerNorm(dims[0], data_format='channels_first'),
        )

        self.downsample_layers = nn.ModuleList([stem])
        # 4 downsampling stages: stride-2 3D conv
        for i in range(len(dims) - 1):
            self.downsample_layers.append(nn.Sequential(
                LayerNorm(dims[i], data_format='channels_first'),
                nn.Conv3d(dims[i], dims[i+1], kernel_size=2, stride=2),
            ))

        # Residual blocks per stage
        # After stage 3 (i=3), temporal dim is 16/2^4=1 -> squeeze to 2D
        self.res_blocks = nn.ModuleList()
        for i in range(len(dims)):
            ndim = 3 if i < len(dims) - 1 else 2
            self.res_blocks.append(nn.Sequential(
                *[ResidualBlock(dims[i], num_spatial_dims=ndim)
                  for _ in range(num_res_blocks[i])]
            ))

        self.dims = list(dims)

    def forward(self, x):
        # x: (B, C, T, H, W)
        for i in range(len(self.dims)):
            x = self.downsample_layers[i](x)
            x = x.squeeze(2)                        # collapse temporal dim when it hits 1
            x = self.res_blocks[i](x)
        return x                                    # (B, dims[-1], H', W')


class ConvPredictor(nn.Module):
    """Small 2D CNN predictor: context embedding -> predicted target embedding."""
    def __init__(self, dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(dim, dim * 2, kernel_size=2, padding=1),
            ResidualBlock(dim * 2, num_spatial_dims=2),
            nn.Conv2d(dim * 2, dim, kernel_size=2),
        )

    def forward(self, x):
        return self.conv(x)


def build_model(cfg):
    encoder = ConvEncoder(
        in_chans=cfg.num_channels,
        dims=cfg.dims,
        num_res_blocks=cfg.num_res_blocks,
        num_frames=cfg.num_frames,
    )
    predictor = ConvPredictor(dim=cfg.dims[-1])
    return encoder, predictor


def count_params(model, label=''):
    n = sum(p.numel() for p in model.parameters())
    t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  {label:20s} total={n:>10,}  trainable={t:>10,}')
    return n


enc, pred = build_model(cfg)
count_params(enc,  'Encoder')
count_params(pred, 'Predictor')
total = sum(p.numel() for p in enc.parameters()) + sum(p.numel() for p in pred.parameters())
print(f'  Total: {total:,}')
assert total < 100_000_000
del enc, pred


  Encoder              total= 2,468,720  trainable= 2,468,720
  Predictor            total=   801,664  trainable=   801,664
  Total: 3,270,384


## VICReg Loss


In [6]:
def off_diagonal(m):
    n = m.shape[0]
    return m.flatten()[:-1].view(n - 1, n + 1)[:, 1:].flatten()


def vicreg_loss(x, y, sim_coeff=2., std_coeff=40., cov_coeff=2., n_chunks=5):
    """
    VICReg loss on 2D feature maps.
    x, y: (B, C, H, W) — encoder outputs for context and target.
    """
    # If 2D maps, add dummy time dim for generality
    if x.ndim == 4:
        x = x.unsqueeze(2)  # (B, C, 1, H, W)
        y = y.unsqueeze(2)

    # Flatten spatial: (B, C, T, H, W) -> (N, C)
    x = x.permute(0, 2, 3, 4, 1).reshape(-1, x.shape[1])  # (B*T*H*W, C)
    y = y.permute(0, 2, 3, 4, 1).reshape(-1, y.shape[1])

    N, C = x.shape

    # Shuffle and chunk for memory efficiency
    idx = torch.randperm(N, device=x.device)
    x = x[idx]
    y = y[idx]

    total_loss = 0.
    total_repr = 0.
    total_std  = 0.
    total_cov  = 0.

    x_chunks = x.chunk(n_chunks)
    y_chunks = y.chunk(n_chunks)

    for xc, yc in zip(x_chunks, y_chunks):
        n = xc.shape[0]

        # Invariance (MSE)
        repr_loss = F.mse_loss(xc, yc)

        # Variance
        xc_c = xc - xc.mean(0)
        yc_c = yc - yc.mean(0)
        std_x = torch.sqrt(xc_c.var(0, unbiased=False) + 1e-4)
        std_y = torch.sqrt(yc_c.var(0, unbiased=False) + 1e-4)
        std_loss = (F.relu(1. - std_x).mean() + F.relu(1. - std_y).mean()) / 2.

        # Covariance
        cov_x = (xc_c.T @ xc_c) / max(1, n - 1)
        cov_y = (yc_c.T @ yc_c) / max(1, n - 1)
        cov_loss = (off_diagonal(cov_x).pow(2).sum() / C +
                    off_diagonal(cov_y).pow(2).sum() / C)

        loss = sim_coeff * repr_loss + std_coeff * std_loss + cov_coeff * cov_loss
        total_loss += loss
        total_repr += repr_loss
        total_std  += std_loss
        total_cov  += cov_loss

    nc = len(x_chunks)
    return {
        'loss':      total_loss / nc,
        'repr_loss': total_repr / nc,
        'std_loss':  total_std  / nc,
        'cov_loss':  total_cov  / nc,
    }

print('VICReg loss defined.')


VICReg loss defined.


## LR Schedule


In [7]:
def cosine_schedule(base_val, final_val, total_steps, warmup_steps=0, start_warmup_val=None):
    if start_warmup_val is None:
        start_warmup_val = final_val
    warmup = np.linspace(start_warmup_val, base_val, warmup_steps) if warmup_steps > 0 else np.array([])
    remain = total_steps - warmup_steps
    if remain > 0:
        t = np.arange(remain) / max(1, remain - 1)
        decay = final_val + 0.5 * (base_val - final_val) * (1 + np.cos(np.pi * t))
    else:
        decay = np.array([])
    return np.concatenate([warmup, decay]).astype(np.float32)

print('Schedule defined.')


Schedule defined.


## Training Loop


In [8]:
def train_one_epoch(encoder, predictor, loader, optimizer, lr_schedule,
                    global_step, epoch, cfg, scaler=None):
    encoder.train()
    predictor.train()
    device = torch.device(cfg.device)
    use_amp = (cfg.device == 'cuda')

    grad_accum = max(cfg.target_global_batch_size // cfg.batch_size, 1)
    running = defaultdict(float)
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        # LR schedule
        idx = min(global_step, len(lr_schedule) - 1)
        for pg in optimizer.param_groups:
            pg['lr'] = float(lr_schedule[idx])

        ctx = batch['context'].to(device, non_blocking=True)  # (B, C, T, H, W)
        tgt = batch['target'].to(device, non_blocking=True)

        with torch.amp.autocast('cuda', enabled=use_amp):
            ctx_embed = encoder(ctx)       # (B, D, H', W')
            tgt_embed = encoder(tgt)       # (B, D, H', W')
            pred      = predictor(ctx_embed)
            loss_dict = vicreg_loss(
                pred, tgt_embed,
                sim_coeff=cfg.sim_coeff,
                std_coeff=cfg.std_coeff,
                cov_coeff=cfg.cov_coeff,
            )
            loss = loss_dict['loss'] / grad_accum

        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % grad_accum == 0:
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    list(encoder.parameters()) + list(predictor.parameters()), cfg.clip_grad)
                scaler.step(optimizer)
                scaler.update()
            else:
                nn.utils.clip_grad_norm_(
                    list(encoder.parameters()) + list(predictor.parameters()), cfg.clip_grad)
                optimizer.step()
            optimizer.zero_grad()
            global_step += 1

        for k, v in loss_dict.items():
            running[k] += v.item()

        if step % 100 == 0:
            lr = optimizer.param_groups[0]['lr']
            print(f'  Ep {epoch} | Step {step:5d}/{len(loader)} '
                  f'| loss {loss_dict["loss"].item():.4f} '
                  f'| repr {loss_dict["repr_loss"].item():.4f} '
                  f'| std {loss_dict["std_loss"].item():.4f} '
                  f'| cov {loss_dict["cov_loss"].item():.4f} '
                  f'| lr {lr:.2e}')

    avg = {k: v / len(loader) for k, v in running.items()}
    return avg, global_step

print('Training loop defined.')


Training loop defined.


## Checkpointing


In [9]:
def save_checkpoint(encoder, predictor, optimizer, scaler, epoch, step, cfg):
    Path(cfg.checkpoint_dir).mkdir(parents=True, exist_ok=True)
    path = os.path.join(cfg.checkpoint_dir, f'baseline_epoch{epoch:04d}.pt')
    torch.save({
        'epoch': epoch,
        'global_step': step,
        'encoder': encoder.state_dict(),
        'predictor': predictor.state_dict(),
        'optimizer': optimizer.state_dict(),
        'scaler': scaler.state_dict() if scaler else None,
    }, path)
    print(f'  Saved -> {path}')

def load_checkpoint(encoder, predictor, optimizer, scaler, path, cfg):
    ckpt = torch.load(path, map_location=cfg.device)
    encoder.load_state_dict(ckpt['encoder'])
    predictor.load_state_dict(ckpt['predictor'])
    optimizer.load_state_dict(ckpt['optimizer'])
    if scaler and ckpt.get('scaler'):
        scaler.load_state_dict(ckpt['scaler'])
    print(f'  Loaded epoch {ckpt["epoch"]}, step {ckpt["global_step"]}')
    return ckpt['epoch'], ckpt['global_step']

def find_latest(d):
    Path(d).mkdir(parents=True, exist_ok=True)
    ckpts = sorted(Path(d).glob('baseline_epoch*.pt'))
    return str(ckpts[-1]) if ckpts else None

print('Checkpoint utils defined.')


Checkpoint utils defined.


## Feature Extraction + Evaluation


In [10]:
@torch.no_grad()
def extract_features(encoder, loader, cfg):
    """
    Run the frozen encoder on context frames and global-avg-pool.
    Returns features (N, D), alphas (N,), zetas (N,).
    """
    encoder.eval()
    device = torch.device(cfg.device)
    feats, alphas, zetas = [], [], []

    for batch in loader:
        ctx = batch['context'].to(device)              # (B, C, T, H, W)
        emb = encoder(ctx)                             # (B, D, H', W')
        emb = emb.mean(dim=(-2, -1))                   # (B, D) global avg pool
        feats.append(emb.cpu().float().numpy())
        params = batch['physical_params']               # (B, 2) = [alpha, zeta]
        alphas.append(params[:, 0].numpy())
        zetas.append(params[:, 1].numpy())

    return np.concatenate(feats), np.concatenate(alphas), np.concatenate(zetas)


def linear_probe_eval(tr_f, tr_a, tr_z, va_f, va_a, va_z):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(tr_f)
    X_va = scaler.transform(va_f)
    results = {}
    for name, y_tr_raw, y_va_raw in [('alpha', tr_a, va_a), ('zeta', tr_z, va_z)]:
        mu, sig = y_tr_raw.mean(), y_tr_raw.std() + 1e-8
        y_tr, y_va = (y_tr_raw - mu) / sig, (y_va_raw - mu) / sig
        reg = Ridge(alpha=1.0).fit(X_tr, y_tr)
        mse = mean_squared_error(y_va, reg.predict(X_va))
        results[f'lp_mse_{name}'] = mse
        print(f'  [Linear Probe] {name:5s}  MSE = {mse:.4f}')
    return results


def knn_eval(tr_f, tr_a, tr_z, va_f, va_a, va_z, k=20):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(tr_f)
    X_va = scaler.transform(va_f)
    results = {}
    for name, y_tr_raw, y_va_raw in [('alpha', tr_a, va_a), ('zeta', tr_z, va_z)]:
        mu, sig = y_tr_raw.mean(), y_tr_raw.std() + 1e-8
        y_tr, y_va = (y_tr_raw - mu) / sig, (y_va_raw - mu) / sig
        knn = KNeighborsRegressor(n_neighbors=k, metric='cosine', n_jobs=-1)
        knn.fit(X_tr, y_tr)
        mse = mean_squared_error(y_va, knn.predict(X_va))
        results[f'knn_mse_{name}'] = mse
        print(f'  [kNN k={k}]      {name:5s}  MSE = {mse:.4f}')
    return results


print('Evaluation functions defined.')


Evaluation functions defined.


## Main


In [11]:
def main(cfg=cfg):
    set_seed(cfg.seed)
    device = torch.device(cfg.device)
    print(f'\n{"="*60}')
    print(f'Baseline JEPA  |  active_matter  |  device: {device}')
    print(f'{"="*60}\n')

    # Data
    train_loader, val_loader = build_dataloaders(cfg)
    grad_accum = max(cfg.target_global_batch_size // cfg.batch_size, 1)
    steps_per_epoch = len(train_loader) // grad_accum
    total_steps = cfg.epochs * steps_per_epoch
    warmup_steps = cfg.warmup_epochs * steps_per_epoch
    print(f'Steps/epoch: {steps_per_epoch}, Total: {total_steps}, Warmup: {warmup_steps}')

    # Model
    encoder, predictor = build_model(cfg)
    encoder.to(device)
    predictor.to(device)
    count_params(encoder, 'Encoder')
    count_params(predictor, 'Predictor')

    # Optimizer
    all_params = list(encoder.parameters()) + list(predictor.parameters())
    optimizer = optim.AdamW(all_params, lr=cfg.lr, weight_decay=cfg.weight_decay, betas=(0.9, 0.95))
    scaler = torch.amp.GradScaler('cuda') if cfg.device == 'cuda' else None

    # LR schedule
    lr_schedule = cosine_schedule(cfg.lr, cfg.min_lr, total_steps,
                                  warmup_steps=warmup_steps, start_warmup_val=cfg.min_lr)

    # Resume
    start_epoch, global_step = 0, 0
    latest = find_latest(cfg.checkpoint_dir)
    if latest:
        start_epoch, global_step = load_checkpoint(
            encoder, predictor, optimizer, scaler, latest, cfg)
        start_epoch += 1

    # WandB
    if cfg.use_wandb and HAS_WANDB:
        import dataclasses
        wandb.init(project=cfg.wandb_project, name='baseline-jepa',
                   config=dataclasses.asdict(cfg), resume='allow')

    # Training
    for epoch in range(start_epoch, cfg.epochs):
        print(f'\n{"─"*60}')
        print(f'Epoch {epoch+1}/{cfg.epochs}')

        losses, global_step = train_one_epoch(
            encoder, predictor, train_loader, optimizer,
            lr_schedule, global_step, epoch, cfg, scaler)
        print(f'  Avg loss: {losses["loss"]:.4f}')

        log_dict = {'epoch': epoch + 1, **{f'train/{k}': v for k, v in losses.items()}}

        # Eval
        if epoch % cfg.eval_interval == 0 or epoch == cfg.epochs - 1:
            print('  Extracting features ...')
            tr_f, tr_a, tr_z = extract_features(encoder, train_loader, cfg)
            va_f, va_a, va_z = extract_features(encoder, val_loader, cfg)
            print('  Linear Probe:')
            lp = linear_probe_eval(tr_f, tr_a, tr_z, va_f, va_a, va_z)
            print('  kNN:')
            kn = knn_eval(tr_f, tr_a, tr_z, va_f, va_a, va_z, k=cfg.knn_k)
            log_dict.update({**lp, **kn})

        if cfg.use_wandb and HAS_WANDB:
            wandb.log(log_dict)

        if epoch % cfg.save_interval == 0 or epoch == cfg.epochs - 1:
            save_checkpoint(encoder, predictor, optimizer, scaler,
                            epoch, global_step, cfg)

    print('\nTraining complete!')
    if cfg.use_wandb and HAS_WANDB:
        wandb.finish()
    return encoder, predictor

print('main() defined.')


main() defined.


In [12]:
# Uncomment to train
encoder, predictor = main(cfg)


Baseline JEPA  |  active_matter  |  device: cuda

train: 8,750 samples from 45 files
valid: 1,200 samples from 16 files
Steps/epoch: 34, Total: 1020, Warmup: 68
  Encoder              total= 2,468,720  trainable= 2,468,720
  Predictor            total=   801,664  trainable=   801,664


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/kk5047/.netrc.
wandb: Currently logged in as: kishlay-kumar1995 (kkumar1995-new-york-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



────────────────────────────────────────────────────────────
Epoch 1/30


/home/kk5047/.local/lib/python3.9/site-packages/torch/nn/modules/conv.py:712: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1027.)
  return F.conv3d(


  Ep 0 | Step     0/1093 | loss 26.5068 | repr 0.3636 | std 0.6416 | cov 0.0587 | lr 1.00e-06
  Ep 0 | Step   100/1093 | loss 26.3559 | repr 0.3683 | std 0.6375 | cov 0.0599 | lr 4.57e-05
  Ep 0 | Step   200/1093 | loss 25.8282 | repr 0.3756 | std 0.6233 | cov 0.0720 | lr 9.05e-05
  Ep 0 | Step   300/1093 | loss 25.0384 | repr 0.4024 | std 0.6012 | cov 0.0937 | lr 1.35e-04
  Ep 0 | Step   400/1093 | loss 23.8903 | repr 0.4588 | std 0.5635 | cov 0.2157 | lr 1.80e-04
  Ep 0 | Step   500/1093 | loss 22.4924 | repr 0.6100 | std 0.4782 | cov 1.0728 | lr 2.25e-04
  Ep 0 | Step   600/1093 | loss 23.5572 | repr 0.7703 | std 0.3948 | cov 3.1114 | lr 2.69e-04
  Ep 0 | Step   700/1093 | loss 20.6636 | repr 0.7305 | std 0.4168 | cov 1.2647 | lr 3.14e-04
  Ep 0 | Step   800/1093 | loss 19.3427 | repr 0.7514 | std 0.4076 | cov 0.7680 | lr 3.74e-04
  Ep 0 | Step   900/1093 | loss 18.4138 | repr 0.9692 | std 0.3155 | cov 1.9285 | lr 4.18e-04
  Ep 0 | Step  1000/1093 | loss 16.8987 | repr 0.9762 | std 

KeyboardInterrupt: 

## Standalone Evaluation


In [ ]:
def evaluate_checkpoint(ckpt_path, cfg=cfg):
    device = torch.device(cfg.device)
    encoder, _ = build_model(cfg)
    encoder.to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    encoder.load_state_dict(ckpt['encoder'])
    print(f'Loaded epoch {ckpt["epoch"]}')

    train_loader, val_loader = build_dataloaders(cfg)

    print('Extracting train features ...')
    tr_f, tr_a, tr_z = extract_features(encoder, train_loader, cfg)
    print('Extracting val features ...')
    va_f, va_a, va_z = extract_features(encoder, val_loader, cfg)

    print('\nLinear Probe:')
    lp = linear_probe_eval(tr_f, tr_a, tr_z, va_f, va_a, va_z)
    print('\nkNN:')
    kn = knn_eval(tr_f, tr_a, tr_z, va_f, va_a, va_z, k=cfg.knn_k)
    return {**lp, **kn}

# Usage:
# results = evaluate_checkpoint('/scratch/.../baseline_epoch0029.pt')
print('evaluate_checkpoint() defined.')
